<a href="https://colab.research.google.com/github/SuperDataWorld/Python/blob/main/Pulling_Tweets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import os
os.environ["X_BEARER_TOKEN"] = "AAAAAAAAAAAAAAAAAAAAABjs7wEAAAAA7iWd9PJ%2BPd1Z1OUN9r3azbZn5vI%3DRPVzo3bi0fiaFngKE5AQYbiM1AsMlUttKlYvDbX8iG7zLsWmaQ"

In [9]:
token = os.getenv("X_BEARER_TOKEN")

In [10]:
import os
import time
import requests
import pandas as pd

BASE_URL = "https://api.x.com/2/tweets/search/recent"  # Search recent Posts endpoint :contentReference[oaicite:1]{index=1}

def search_recent_posts_to_df(
    bearer_token: str,
    query: str,
    max_pages: int = 5,
    max_results: int = 100,
    sleep_seconds: float = 1.0,
) -> pd.DataFrame:
    """
    Descarga posts (últimos 7 días) que matcheen query, con paginación (next_token),
    y devuelve un DataFrame.
    """
    headers = {"Authorization": f"Bearer {bearer_token}"}  # Bearer auth header :contentReference[oaicite:2]{index=2}

    # Campos recomendados (ajusta a tu necesidad)
    params = {
        "query": query,                 # required :contentReference[oaicite:3]{index=3}
        "max_results": max_results,      # 10..100 :contentReference[oaicite:4]{index=4}
        "tweet.fields": ",".join([       # tweet.fields list :contentReference[oaicite:5]{index=5}
            "id", "text", "created_at", "author_id", "lang", "geo", "public_metrics"
        ]),
        "expansions": ",".join([
            "author_id",
            "geo.place_id"
        ]),
        "user.fields": ",".join([
            "id", "name", "username", "created_at"
        ]),
        "place.fields": ",".join([      # place.fields list :contentReference[oaicite:6]{index=6}
            "id", "full_name", "country", "country_code", "place_type", "geo"
        ]),
        "sort_order": "recency",         # option :contentReference[oaicite:7]{index=7}
    }

    all_rows = []
    next_token = None

    for page in range(max_pages):
        if next_token:
            params["next_token"] = next_token  # pagination token :contentReference[oaicite:8]{index=8}
        else:
            params.pop("next_token", None)

        r = requests.get(BASE_URL, headers=headers, params=params, timeout=30)
        if r.status_code != 200:
            raise RuntimeError(f"X API error {r.status_code}: {r.text}")

        payload = r.json()

        data = payload.get("data", []) or []
        includes = payload.get("includes", {}) or {}
        meta = payload.get("meta", {}) or {}

        # Indexa "includes" para hacer join fácil
        users_by_id = {u["id"]: u for u in includes.get("users", []) or []}
        places_by_id = {p["id"]: p for p in includes.get("places", []) or []}

        for t in data:
            author = users_by_id.get(t.get("author_id"), {})
            place_id = (t.get("geo") or {}).get("place_id")
            place = places_by_id.get(place_id, {}) if place_id else {}

            all_rows.append({
                "tweet_id": t.get("id"),
                "created_at": t.get("created_at"),
                "text": t.get("text"),
                "lang": t.get("lang"),
                "author_id": t.get("author_id"),
                "username": author.get("username"),
                "name": author.get("name"),
                "place_id": place_id,
                "place_full_name": place.get("full_name"),
                "place_country": place.get("country"),
                "place_country_code": place.get("country_code"),
                "place_type": place.get("place_type"),
                "retweet_count": (t.get("public_metrics") or {}).get("retweet_count"),
                "reply_count": (t.get("public_metrics") or {}).get("reply_count"),
                "like_count": (t.get("public_metrics") or {}).get("like_count"),
                "quote_count": (t.get("public_metrics") or {}).get("quote_count"),
            })

        next_token = meta.get("next_token")
        if not next_token:
            break

        time.sleep(sleep_seconds)

    return pd.DataFrame(all_rows)


if __name__ == "__main__":
    # 1) Pon tu token en variable de entorno:
    # export X_BEARER_TOKEN="..."
    token = os.getenv("X_BEARER_TOKEN")
    if not token:
        raise ValueError("Falta X_BEARER_TOKEN en variables de entorno.")

    # 2) Query: "scam" + EE.UU. (si hay geo), excluyendo retweets
    #    Ojo: place_country:US depende de que el tweet tenga place/geo.
    query = 'scam place_country:US -is:retweet'

    df = search_recent_posts_to_df(
        bearer_token=token,
        query=query,
        max_pages=10,
        max_results=100
    )

    # 3) Guardar el resultado
    df.to_csv("scam_us_recent_posts.csv", index=False, encoding="utf-8")
    # o mejor para análisis: df.to_parquet("scam_us_recent_posts.parquet", index=False)

    print(df.head())
    print(f"Total filas: {len(df)}")

RuntimeError: X API error 402: {"account_id":2028447877882867712,"title":"CreditsDepleted","detail":"Your enrolled account [2028447877882867712] does not have any credits to fulfill this request.","type":"https://api.twitter.com/2/problems/credits"}

In [ ]:
"""
❗ Realidad en 2026

La API de X ya no es verdaderamente gratuita como antes.
El plan free:

Tiene límites muy estrictos

O directamente no permite el endpoint search/recent

Y cuando se acaban los créditos → error 402

“All developers” no significa gratis ilimitado.
Significa que el endpoint está disponible para todos los planes… pero dentro de los créditos de tu plan.
"""


In [ ]:
tweets_df

,User,Date Created,Number of Likes,Source of Tweet,Tweet
0,Muswema Mukuni Kambaki,2022-12-23 04:38:40+00:00,0,Twitter for Android,"Once again, the English are criticising everyo..."
1,Alia Liverpool,2022-12-22 22:07:11+00:00,0,Twitter for Android,Btw the quality of the ref in England is so ba...
2,United Flag Designs,2022-12-22 21:29:44+00:00,0,Twitter for iPhone,This ref has clearly watched the World Cup and...
3,Jared Ogle,2022-12-22 21:27:11+00:00,0,Twitter for Android,A world cup ref gives that penalty.
4,LeeJ,2022-12-22 20:36:25+00:00,0,Twitter for Android,This ref didn't watch any of the World Cup.......
...,...,...,...,...,...
95,megan,2022-12-18 18:24:51+00:00,0,Twitter for Android,they should make this bit of the world cup mor...
96,Plough Lane By Numbers,2022-12-18 18:24:16+00:00,3,Twitter for Android,Incredible World Cup final (obvs). 🇦🇷 worthy w...
97,⚽️Football Oracle⚽️,2022-12-18 18:21:59+00:00,0,Twitter for iPhone,The World Cup ref was amazing. #ArgentinaVsFra...
98,Richard Porteous,2022-12-18 18:21:17+00:00,4,Twitter for Android,Thought I would watch some of the football wor...


In [ ]:
tweets[0]

Status(_api=<tweepy.api.API object at 0x7f2c587c0b80>, _json={'created_at': 'Fri Dec 23 04:38:40 +0000 2022', 'id': 1606147252468211714, 'id_str': '1606147252468211714', 'full_text': 'Once again, the English are criticising everyone in their world cup defeat except the guy who missed a penalty that would have taken the game to extra time. They blame the ref, FIFA, Rashford, climate change but not Mr England (Kane) who hoofed that ball into the stands.', 'truncated': False, 'display_text_range': [0, 271], 'entities': {'hashtags': [], 'symbols': [], 'user_mentions': [], 'urls': []}, 'metadata': {'iso_language_code': 'en', 'result_type': 'recent'}, 'source': '<a href="http://twitter.com/download/android" rel="nofollow">Twitter for Android</a>', 'in_reply_to_status_id': None, 'in_reply_to_status_id_str': None, 'in_reply_to_user_id': None, 'in_reply_to_user_id_str': None, 'in_reply_to_screen_name': None, 'user': {'id': 1267956633830862850, 'id_str': '1267956633830862850', 'name': 'Muswema M

In [ ]:
tweets_df.to_csv('tweets.csv') 
files.download('tweets.csv')